# logit-thermo ゲート前の材料集め（G-pre）：場をかけたことのない 7 本のモデルで、8.30 と同じ場の生成（q つき）——ランタイム 1/3（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。
- **3 本のノートブック（ランタイム 1/3・2/3・3/3）を、別々のランタイムで同時に回してください。** 21 個の仕事（7 モデル × h ＝ −0.3・0・＋0.3）を、3 つのランタイムが「まだ終わっていない・誰も取っていない」ものから取りに行きます。3 つ同時なら 3 時間くらいで全部終わる見込みです。1 つだけでも、時間はかかりますが全部回せます。
- 作り：温度 0.4〜0.7 の 7 個、記事 40 × サンプル 4、512 歩。各歩の q（候補の確率の和）とエネルギー U も記録します。モデル：Pythia-410M・Pythia-1.4B・GPT-Neo-1.3B・SmolLM2-1.7B・Qwen2.5-1.5B・GPT-2-XL・BLOOM-1.1B。
- **データは、ゲートを凍結するまで開けません。** 結果は表示しません。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコード、`scripts/a2_explore_field.py`（`0ac42c7ab82b06d2…`）、`scripts/a2_explore_dsc.py`（`33f47a0a3bbda183…`）、`scripts/a2_explore_dsc2.py`（`cddae78690360418…`）は Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_gpre_collect.py`（`49aa63e0c3479f98…`）はこのノートブックに埋め込み。
- 保存先 `A2_gpre/`。途中で切れても、もう一度「すべて実行」すれば、終わった仕事は飛ばします（取りかけの仕事は 2 時間たつとほかのランタイムが取り直せます）。
- **通知**：🚀・仕事ごとの ✅・🎉（このランタイムで回した数と、まだ残っている仕事）・⚠️。名札は「logit-thermo ゲート前の材料集め G-pre（7 モデル × 場 ±0.3、q つき、ランタイム 1/3、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo ゲート前の材料集め G-pre（7 モデル × 場 ±0.3、q つき、ランタイム 1/3、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - G-pre collect worker 1of3', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', 'SRCD2': 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD, SRCD2 = _s['SRCD'], _s['SRCD2']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCG = '"""ゲートの前の材料集め（G-pre）：8.30 の場の下見と同じ作り（温度 0.4〜0.7 の 7 個、h ＝ −0.3・0・＋0.3、記事 40 × サンプル 4、512 歩）を、\n場をかけたことのない 7 本のモデルで回し、各歩の q も記録する（2026-10-31 20:1x、GPU（L4）を 3 つ、判定しない）。\nデータは、ゲート（G0 ものさし・G-エンタルピーの整合）を凍結するまで開けない。\n\nscripts/a2_2_collect.py・a2_4_collect.py・a2_7_collect.py・a2_7b_collect.py・a2_explore_field.py・a2_explore_dsc.py・a2_explore_dsc2.py を\n実行したあとの同じ名前空間で実行する（load_model・valid_mask・split_doc・CONT・D_CHUNK・generate_dsc2 を変えずに使う）。\n生成は generate_dsc2 に一定の温度と一定の場を渡す（generate_field と同じトークンになることは self_test_dsc2 で確かめてある）。\n記録：トークン、候補があったか、候補を選んだか、U（温度 1・場なし）、q0（場の前）、q（場の後）。形は (温度 7, サンプル 4, 記事, 512)。\n\n落ちた仕事：エラーの全文を ⚠ で通知し、印（locks/…failed）をつけて飛ばし、ほかの仕事を続ける（ひと晩回すため）。\n仕事の割り振り：21 個の仕事（7 モデル × 3 つの h）を、3 つのランタイムが「まだ終わっていない・誰も取っていない」ものから取りに行く。\n取ったしるしは Drive のロックのファイル（2 時間たったら古いとみなして取り直してよい）。種は仕事だけで決まる（どのランタイムが回しても同じ結果）ので、\n2 つのランタイムが同じ仕事をしてしまっても、同じ中身を書くだけで、結果は変わらない。\n"""\nimport gc, os, time, json, socket\nimport numpy as np\nimport torch\n\nGP_TEMPS = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]\nGP_H = [(\'hm03\', -0.3), (\'h0\', 0.0), (\'hp03\', 0.3)]\nGP_S = 4\nGP_SEED = 20261141\nGP_MODELS = [(\'pythia-410m\', \'EleutherAI/pythia-410m\'), (\'pythia-1.4b\', \'EleutherAI/pythia-1.4b\'), (\'gpt-neo-1.3b\', \'EleutherAI/gpt-neo-1.3B\'),\n             (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'), (\'gpt2-1.5b\', \'openai-community/gpt2-xl\'),\n             (\'bloom-1b1\', \'bigscience/bloom-1b1\')]\nGP_CHUNK = 20\nGP_STALE = 2 * 3600\n\n\ndef gp_seed(k, s, c):\n    """温度の番号 k、サンプル s、記事のかたまり c → 種（3 つの h で同じ ＝ 共通の乱数）。"""\n    return GP_SEED + 1000 * k + 10 * s + c\n\n\ndef gp_jobs():\n    return [(name, hf, arm, h) for name, hf in GP_MODELS for arm, h in GP_H]\n\n\ndef gp_claim(outdir, name, arm, worker):\n    """まだ終わっていなくて、誰も取っていない（または古い）なら取る → True。"""\n    if os.path.exists(f\'{outdir}/{name}@{arm}.npz\') or os.path.exists(f\'{outdir}/locks/{name}@{arm}.failed\'):\n        return False\n    lk = f\'{outdir}/locks/{name}@{arm}.lock\'\n    if os.path.exists(lk):\n        try:\n            age = time.time() - json.load(open(lk))[\'t\']\n        except Exception:\n            age = 0.0\n        if age < GP_STALE:\n            return False\n    os.makedirs(f\'{outdir}/locks\', exist_ok=True)\n    json.dump(dict(worker=worker, host=socket.gethostname(), t=time.time()), open(lk, \'w\'))\n    time.sleep(20)   # Drive の同期を少し待ってから、自分のしるしのままか確かめる\n    try:\n        return json.load(open(lk)).get(\'worker\') == worker\n    except Exception:\n        return False\n\n\ndef run_gpre_job(name, hf_id, arm, h, docs, outdir, device, cache):\n    """1 つの仕事（モデル × h）。cache：同じモデルを続けて使うときに読み直さないための入れ物。"""\n    if cache.get(\'name\') != name:\n        cache.clear(); gc.collect(); torch.cuda.empty_cache()\n        tok, model = load_model(hf_id, device)\n        assert next(model.parameters()).dtype == torch.float32\n        valid, eos = valid_mask(tok, model, device)\n        pairs = [split_doc(tok, d[\'text\']) for d in docs]\n        use = [i for i, p in enumerate(pairs) if p is not None]\n        P = {len(pairs[i][0]) for i in use}\n        assert len(P) == 1\n        maxpos = getattr(model.config, \'max_position_embeddings\', None) or getattr(model.config, \'n_positions\', None)\n        assert maxpos is None or list(P)[0] + CONT <= maxpos, (\'文が長すぎます\', maxpos)\n        cache.update(name=name, model=model, valid=valid, use=use, prompts=[pairs[i][0] for i in use])\n    model, valid, use, prompts = cache[\'model\'], cache[\'valid\'], cache[\'use\'], cache[\'prompts\']\n    nD = len(use)\n    chunks = [list(range(i, min(nD, i + GP_CHUNK))) for i in range(0, nD, GP_CHUNK)]\n    ta = time.time()\n    dt = dict(toks=np.int32, cand=np.bool_, pick=np.bool_, U=np.float32, q0=np.float32, q=np.float32)\n    R = {k: np.zeros((len(GP_TEMPS), GP_S, nD, CONT), v) for k, v in dt.items()}\n    for k, T in enumerate(GP_TEMPS):\n        Ts, Hs = np.full(CONT, T), np.full(CONT, h)\n        for s in range(GP_S):\n            for c, ii in enumerate(chunks):\n                g = torch.Generator(device=device).manual_seed(gp_seed(k, s, c))\n                r = generate_dsc2(model, [prompts[i] for i in ii], Ts, Hs, valid, g)\n                for kk in R:\n                    R[kk][k, s, ii] = r[kk]\n                torch.cuda.empty_cache()\n    path = f\'{outdir}/{name}@{arm}.npz\'\n    np.savez_compressed(path + \'.part.npz\', temps=np.array(GP_TEMPS), h=h, arm=arm, use=np.array(use), seconds=time.time() - ta, chunk=GP_CHUNK, **R)\n    os.replace(path + \'.part.npz\', path)\n    return time.time() - ta\n\n\ndef run_gpre(worker, docs, outdir, device, notify=print):\n    """worker（0・1・2）：自分の番号ぶんずらした順に仕事を見て、取れるものを回す。取れる仕事がなくなったら終わる。"""\n    os.makedirs(outdir, exist_ok=True)\n    jobs = gp_jobs()\n    order = jobs[7 * worker:] + jobs[:7 * worker]\n    cache, n, t0 = {}, 0, time.time()\n    while True:\n        took = False\n        for name, hf, arm, h in order:\n            if gp_claim(outdir, name, arm, worker):\n                took = True\n                try:\n                    sec = run_gpre_job(name, hf, arm, h, docs, outdir, device, cache)\n                    n += 1\n                    notify(f\'{name}@{arm} おわり（{sec:.0f} 秒、このランタイムで {n} 個目、通算 {time.time() - t0:.0f} 秒）\')\n                except Exception as e:\n                    # 1 つの仕事が落ちても、ほかの仕事は続ける。落ちた仕事には印をつけて、ほかのランタイムも取らないようにする\n                    import traceback\n                    txt = \'\'.join(traceback.format_exception(type(e), e, e.__traceback__))\n                    v = os.environ.get(\'HF_TOKEN\')\n                    if v:\n                        txt = txt.replace(v, \'***\')\n                    txt = txt.encode(\'utf-8\')[-3000:].decode(\'utf-8\', \'ignore\')\n                    json.dump(dict(worker=worker, t=time.time(), error=txt), open(f\'{outdir}/locks/{name}@{arm}.failed\', \'w\'), ensure_ascii=False)\n                    notify(f\'⚠️ {name}@{arm} で止まりました（この仕事は飛ばして、ほかを続けます）：\\n\' + txt)\n                    cache.clear(); gc.collect(); torch.cuda.empty_cache()\n                break\n        if not took:\n            break\n    cache.clear(); gc.collect(); torch.cuda.empty_cache()\n    left = [f\'{nm}@{a}\' + (\'（失敗）\' if os.path.exists(f\'{outdir}/locks/{nm}@{a}.failed\') else \'\') for nm, _, a, _ in jobs if not os.path.exists(f\'{outdir}/{nm}@{a}.npz\')]\n    return n, left\n\n\ndef self_test_gpre():\n    assert len(gp_jobs()) == 21 and len({(a, b) for a, _, b, _ in gp_jobs()}) == 21\n    seeds = {gp_seed(k, s, c) for k in range(7) for s in range(GP_S) for c in range(2)}\n    assert len(seeds) == 7 * GP_S * 2\n    # ずらした順はどれも 21 個の並べかえ\n    jobs = gp_jobs()\n    for w in range(3):\n        o = jobs[7 * w:] + jobs[:7 * w]\n        assert sorted(o) == sorted(jobs)\n    return \'OK\'\n'
assert hashlib.sha256(SRCG.encode()).hexdigest() == '49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
exec(SRCG)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 仕事を取りに行って回す → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'), self_test_dsc2('cuda'), self_test_gpre())
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_gpre', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    N_DONE, LEFT = run_gpre(0, docs, '/content/drive/MyDrive/logit-thermo/A2_gpre', 'cuda', notify=lambda s: notify(s, tags='warning') if s.startswith('⚠') else notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(worker=0, done=N_DONE, left=LEFT, models=GP_MODELS, h=GP_H, temps=GP_TEMPS, samples=GP_S, chunk=GP_CHUNK, seed=GP_SEED,
                   gpre_sha256='49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b', dsc2_sha256='cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_gpre' + '/run_meta_worker0.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ ゲート前の材料集め G-pre（ランタイム 1/3）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify(f'🎉 終わりました：ゲート前の材料集め（ランタイム 1/3）。このランタイムで {N_DONE} 個。まだ残っている仕事：{len(LEFT)} 個' + (f'（{", ".join(LEFT)}）' if LEFT else '（全部そろいました）'), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
